# Part 3 — Missing Values & Outliers
### Data Mondays, Week 4 — Amani Insurance claims case study

**Deliverable:** identify and handle **missing values** and **outliers**.

This is the notebook that does the *full* cleanup that Parts 1 and 2
deliberately kept lightweight. By the end, `clean_claims()` returns a
DataFrame that Parts 4 and 5 build on directly.

In [ ]:
import numpy as np
import pandas as pd

FILE_PATH = "insurance_claims_messy.csv"

## Section 1 — Identify missing values

Before deciding how to fix anything, quantify exactly how much is missing and
where.

In [ ]:
df = pd.read_csv(FILE_PATH)

# Same comma-strip + coercion as Part 1 -- blanks and stray garbage both
# become NaN (pandas' missing-value marker) via errors="coerce".
df["claim_amount_kes"] = (
    df["claim_amount_kes"].astype(str).str.replace(",", "", regex=False).str.strip()
)
df["claim_amount_kes"] = pd.to_numeric(df["claim_amount_kes"], errors="coerce")

# .isna() returns True/False per cell; .sum() on that counts the Trues --
# i.e. this gives us a missing-value count per column in one line.
df.isna().sum()

In [ ]:
pct_missing = df["claim_amount_kes"].isna().mean() * 100
print(f"% of rows missing claim_amount_kes: {pct_missing:.1f}%")

## Section 2 — Deciding HOW to handle missing values

There is no single "correct" fix for a missing value — the right strategy
depends on what you're about to do with the data. Three common options, and
when each makes sense for *this* dataset:

| Strategy | When it's the right call |
|---|---|
| **(a) Drop the row** | Missing amounts are rare, and you only need overall totals/averages |
| **(b) Fill with the overall median** | Keeps every row for other columns, but understates true spread if claim types vary a lot |
| **(c) Fill with the per-claim-type median** | The right choice here — a missing health-outpatient claim and a missing property-fire claim are not on the same scale at all |

In [ ]:
df["claim_type_clean"] = (
    df["claim_type"].astype(str).str.strip().str.lower().str.replace(" ", "-")
)

# (a) Drop every row with a missing amount.
dropped = df.dropna(subset=["claim_amount_kes"])
print(f"(a) Drop: {len(df) - len(dropped)} rows removed, {len(dropped)} remain.")

# (b) Fill every missing value with the SAME overall median.
overall_median = df["claim_amount_kes"].median()
filled_overall = df["claim_amount_kes"].fillna(overall_median)
print(f"(b) Fill with overall median (KES {overall_median:,.0f}).")

# (c) Fill each row's missing value with THAT ROW'S claim-type median.
# groupby(...).transform("median") returns one value per ROW (not per group),
# so it lines up directly with df's own index and can be passed to fillna().
group_median = df.groupby("claim_type_clean")["claim_amount_kes"].transform("median")
filled_by_group = df["claim_amount_kes"].fillna(group_median)
print("(c) Fill with per-claim-type median -- e.g. a missing health-outpatient")
print("    claim gets filled with the health-outpatient median, not the")
print("    whole-dataset median.")

We'll use strategy **(c)** in `clean_claims()` below — it's the most defensible
choice here because claim sizes vary by an order of magnitude across claim
types.

## Section 3 — Identify outliers with the IQR method

**IQR** = "interquartile range" = Q3 (75th percentile) minus Q1 (25th
percentile) — the width of the middle 50% of the data. The classic rule of
thumb: anything below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR` counts as a
statistical outlier. This is exactly the rule Seaborn's boxplot whiskers use
in Part 2.

We compute this **per claim type**, not once for the whole dataset, for the
same reason as Section 2 — the claim types aren't on the same scale.

In [ ]:
def iqr_bounds(series, k=1.5):
    """Return (lower, upper) outlier bounds using Q1 - k*IQR / Q3 + k*IQR.
    k=1.5 is the standard boxplot default."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

In [ ]:
# Start with every row marked "not an outlier" (False), then flip specific
# rows to True as we find them, one claim type at a time.
outlier_flags = pd.Series(False, index=df.index)

for claim_type, group in df.groupby("claim_type_clean"):
    valid = group["claim_amount_kes"].dropna()
    if len(valid) < 5:      # too few data points to trust a quartile calculation
        continue
    lower, upper = iqr_bounds(valid)
    is_outlier = (group["claim_amount_kes"] < lower) | (group["claim_amount_kes"] > upper)
    outlier_flags.loc[group.index] = is_outlier.fillna(False)
    n_out = is_outlier.sum()
    if n_out:
        print(f"{claim_type:20s} bounds=({lower:>10,.0f}, {upper:>12,.0f})  outliers found: {n_out}")

df["is_outlier"] = outlier_flags
print(f"\nTotal rows flagged as outliers: {df['is_outlier'].sum()} of {len(df)}")

## Section 4 — Handling outliers: flag, don't silently delete

Deleting every flagged outlier is usually the wrong first move — some outliers
are completely genuine (a real KES 4,000,000 fire claim isn't an error, it's the
whole point of having fire cover). The safer default:

1. **Flag** them (done above, in `is_outlier`)
2. **Look at a sample** before deciding anything
3. Only correct/remove the ones that are clearly **data-entry errors** — caught
   with a sanity rule, not a statistical test.

In [ ]:
# A claim amount can NEVER legitimately be negative -- this is a sanity
# rule specific to the domain, not a statistical test, and it catches
# errors that IQR alone might miss.
negative_mask = df["claim_amount_kes"] < 0
print(f"Rows with a NEGATIVE claim_amount_kes (unambiguous errors): {negative_mask.sum()}")
df.loc[negative_mask, ["claim_id", "claim_type_clean", "claim_amount_kes"]].head()

In [ ]:
# Compare: these are flagged by STATISTICS, but aren't necessarily wrong --
# each one needs a human look, not an automatic delete.
df.loc[df["is_outlier"] & ~negative_mask,
       ["claim_id", "claim_type_clean", "claim_amount_kes"]].head()

## Section 5 — `clean_claims()`: the reusable pipeline function

Everything above, packaged into one function that later notebooks (Parts 4
and 5) can call directly instead of repeating all this cleaning by hand.

In [ ]:
def clean_claims(filepath):
    """Load and clean the Amani claims CSV, returning a DataFrame ready
    for analysis. Fixes applied:
        - claim_amount_kes: comma-stripped, coerced to numeric
        - negative amounts: treated as sign-entry errors -> made positive
        - missing amounts: filled with that claim type's median
        - claim_type: canonicalised to lower-case, hyphenated form
        - status/region: stripped + canonicalised casing
        - an `is_outlier` flag (IQR method, per claim type) is kept as a
          column rather than used to silently drop rows -- downstream
          analysis can decide whether to exclude flagged rows or not.
    """
    data = pd.read_csv(filepath)

    data["claim_amount_kes"] = (
        data["claim_amount_kes"].astype(str).str.replace(",", "", regex=False).str.strip()
    )
    data["claim_amount_kes"] = pd.to_numeric(data["claim_amount_kes"], errors="coerce")

    # Negative amounts are a sign-entry error in this domain -- flip them positive.
    data["claim_amount_kes"] = data["claim_amount_kes"].abs()

    # Canonicalise the categorical columns.
    data["claim_type"] = (
        data["claim_type"].astype(str).str.strip().str.lower().str.replace(" ", "-")
    )
    data["status"] = data["status"].astype(str).str.strip().str.lower().str.replace("-", " ")
    data["region"] = data["region"].astype(str).str.strip().str.title()

    # Fill missing amounts with the PER-CLAIM-TYPE median (Section 2, strategy c).
    group_median = data.groupby("claim_type")["claim_amount_kes"].transform("median")
    data["claim_amount_kes"] = data["claim_amount_kes"].fillna(group_median)

    # Drop exact duplicate rows (the same claim logged twice by mistake).
    before = len(data)
    data = data.drop_duplicates()
    print(f"dropped {before - len(data)} exact duplicate rows")

    # Flag outliers per claim type -- kept as a column, never used to delete rows.
    flags = pd.Series(False, index=data.index)
    for claim_type, group in data.groupby("claim_type"):
        lower, upper = iqr_bounds(group["claim_amount_kes"])
        is_out = (group["claim_amount_kes"] < lower) | (group["claim_amount_kes"] > upper)
        flags.loc[group.index] = is_out
    data["is_outlier"] = flags

    return data

In [ ]:
cleaned = clean_claims(FILE_PATH)

print(f"\nFinal cleaned shape: {cleaned.shape}")
print(f"Remaining missing claim_amount_kes: {cleaned['claim_amount_kes'].isna().sum()}")
print(f"Remaining negative claim_amount_kes: {(cleaned['claim_amount_kes'] < 0).sum()}")
print(f"Outliers flagged (kept, not dropped): {cleaned['is_outlier'].sum()}")

**Up next:** Part 4 uses NumPy directly on this cleaned data to show why
vectorised operations are so much faster than writing your own loops.